# 04. OD 매트릭스 분석

티머니 STIS 택시 운행 데이터(D012)를 활용하여 행정동 기반 OD(Origin-Destination) 매트릭스를 분석한다.

**분석 항목:**
1. 행정동코드 기반 OD 매트릭스 생성
2. 상위 OD 쌍 Top 20 바차트
3. 시간대별 OD 패턴 변화 히트맵
4. 요일별 주요 OD 변화
5. 유입/유출 불균형 분석
6. OD 거리 분포
7. 요약 테이블

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# 폰트 설정 (Windows)
plt.rcParams['font.family'] = 'Malgun Gothic'
# Mac 사용 시 아래 줄로 교체
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

print('라이브러리 로드 완료')

In [ ]:
# === 메모리 최적화 유틸 ===
import gc, psutil, os

def mem_usage():
    """현재 RAM 사용량 출력"""
    gb = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    print(f'RAM: {gb:.1f} GB')

def optimize_dtypes(df, cat_cols=None):
    """DataFrame 메모리 최적화"""
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = df[col].astype('int32')
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    if cat_cols:
        for col in cat_cols:
            if col in df.columns:
                df[col] = df[col].astype('category')
    return df

CHUNK_SIZE = 1_000_000  # D012 chunk 크기
D012_CAT_COLS = ['RIDE_A_CD', 'ALIGHT_A_CD', 'DRIVER_ID', 'TAXI_VEHC_ID', 'TRANSP_BIZR_ID']

mem_usage()

In [ ]:
# D012 chunk 로드 (메모리 최적화)
usecols = ['RIDE_DTIME', 'PAY_AMT', 'RIDE_DIST', 'RIDE_A_CD', 'ALIGHT_A_CD']
dtypes = {'RIDE_DTIME': str, 'PAY_AMT': 'int32', 'RIDE_DIST': 'int32'}

od_agg_list = []
hourly_od_list = []
total = 0

for chunk in pd.read_csv('./DC_TBYXD012.csv', usecols=usecols, dtype=dtypes, chunksize=CHUNK_SIZE):
    chunk['ride_dt'] = pd.to_datetime(chunk['RIDE_DTIME'], format='%Y%m%d%H%M%S', errors='coerce')
    chunk = chunk.dropna(subset=['ride_dt'])
    chunk['hour'] = chunk['ride_dt'].dt.hour
    chunk['weekday'] = chunk['ride_dt'].dt.dayofweek
    chunk['day_name'] = chunk['ride_dt'].dt.day_name()
    
    # 시간대 구분
    def timeband(h):
        if 7 <= h <= 9: return '출근(7-9)'
        elif 11 <= h <= 13: return '점심(11-13)'
        elif 17 <= h <= 19: return '퇴근(17-19)'
        elif h >= 23 or h <= 4: return '심야(23-4)'
        else: return '기타'
    chunk['timeband'] = chunk['hour'].apply(timeband)
    chunk['OD'] = chunk['RIDE_A_CD'].astype(str) + '→' + chunk['ALIGHT_A_CD'].astype(str)
    
    total += len(chunk)
    
    # OD 쌍별 집계
    od_agg_list.append(chunk.groupby(['RIDE_A_CD', 'ALIGHT_A_CD']).agg(
        trip_count=('PAY_AMT', 'count'),
        avg_dist=('RIDE_DIST', 'mean'),
        avg_fare=('PAY_AMT', 'mean')
    ).reset_index())
    
    # 시간대/요일별 OD
    hourly_od_list.append(chunk.groupby(['OD', 'timeband', 'day_name']).size().reset_index(name='count'))
    
    del chunk
    gc.collect()

# 합산
od_matrix = pd.concat(od_agg_list).groupby(['RIDE_A_CD', 'ALIGHT_A_CD']).agg(
    trip_count=('trip_count', 'sum'),
    avg_dist=('avg_dist', 'mean'),
    avg_fare=('avg_fare', 'mean')
).reset_index()
od_matrix['OD'] = od_matrix['RIDE_A_CD'].astype(str) + '→' + od_matrix['ALIGHT_A_CD'].astype(str)

hourly_od = pd.concat(hourly_od_list).groupby(['OD', 'timeband', 'day_name'])['count'].sum().reset_index()

del od_agg_list, hourly_od_list
gc.collect()

print(f'전체 건수: {total:,}')
print(f'OD 쌍 수: {len(od_matrix):,}')
mem_usage()

In [ ]:
# 시간 컬럼 파싱
df['RIDE_DTIME'] = pd.to_datetime(df['RIDE_DTIME'], errors='coerce')
df['ALIGHT_DTIME'] = pd.to_datetime(df['ALIGHT_DTIME'], errors='coerce')

# 파생 컬럼
df['hour'] = df['RIDE_DTIME'].dt.hour
df['dayofweek'] = df['RIDE_DTIME'].dt.dayofweek  # 0=월 ~ 6=일
df['day_name'] = df['RIDE_DTIME'].dt.day_name()

# OD 키
df['RIDE_A_CD'] = df['RIDE_A_CD'].astype(str)
df['ALIGHT_A_CD'] = df['ALIGHT_A_CD'].astype(str)
df['OD'] = df['RIDE_A_CD'] + ' -> ' + df['ALIGHT_A_CD']

# 시간대 구분
def assign_timeband(h):
    if 7 <= h <= 9:
        return '출근(7-9)'
    elif 11 <= h <= 13:
        return '점심(11-13)'
    elif 17 <= h <= 19:
        return '퇴근(17-19)'
    elif h >= 23 or h <= 4:
        return '심야(23-4)'
    else:
        return '기타'

df['timeband'] = df['hour'].apply(assign_timeband)

print('전처리 완료')
print(f"유효 건수: {df['RIDE_DTIME'].notna().sum():,}")

## 1. OD 매트릭스 생성

In [ ]:
# 전체 OD 매트릭스 (피벗)
od_matrix = df.groupby(['RIDE_A_CD', 'ALIGHT_A_CD']).size().reset_index(name='trip_count')

# 상위 행정동만 추출하여 히트맵용 피벗 생성
top_origins = df['RIDE_A_CD'].value_counts().head(15).index
top_dests = df['ALIGHT_A_CD'].value_counts().head(15).index
top_codes = list(set(top_origins) | set(top_dests))[:15]

od_pivot = od_matrix[
    od_matrix['RIDE_A_CD'].isin(top_codes) & od_matrix['ALIGHT_A_CD'].isin(top_codes)
].pivot_table(index='RIDE_A_CD', columns='ALIGHT_A_CD', values='trip_count', fill_value=0).astype(int)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(od_pivot, annot=True, fmt='d', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('OD 매트릭스 (상위 행정동)', fontsize=14)
ax.set_xlabel('하차 행정동코드')
ax.set_ylabel('승차 행정동코드')
plt.tight_layout()
plt.show()

print(f'\n고유 OD 쌍 수: {len(od_matrix):,}')
print(f'총 통행량: {od_matrix["trip_count"].sum():,}')

## 2. 상위 OD 쌍 Top 20

In [ ]:
od_top20 = od_matrix.nlargest(20, 'trip_count').copy()
od_top20['OD'] = od_top20['RIDE_A_CD'] + ' -> ' + od_top20['ALIGHT_A_CD']

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(range(len(od_top20)), od_top20['trip_count'].values, color='steelblue', edgecolor='black', linewidth=0.5)
ax.set_yticks(range(len(od_top20)))
ax.set_yticklabels(od_top20['OD'].values, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('통행 건수')
ax.set_title('상위 OD 쌍 Top 20', fontsize=14)

for i, v in enumerate(od_top20['trip_count'].values):
    ax.text(v + 0.5, i, f'{v:,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 3. 시간대별 OD 패턴 변화 히트맵

In [ ]:
# 시간대별 상위 OD 쌍 통행량
top_od_list = od_top20['OD'].tolist()[:10]  # 상위 10개만

timeband_order = ['출근(7-9)', '점심(11-13)', '퇴근(17-19)', '심야(23-4)', '기타']
df_top_od = df[df['OD'].isin(top_od_list)].copy()

tb_od = df_top_od.groupby(['timeband', 'OD']).size().reset_index(name='count')
tb_pivot = tb_od.pivot_table(index='OD', columns='timeband', values='count', fill_value=0).astype(int)
# 컬럼 순서 정렬
tb_pivot = tb_pivot.reindex(columns=[c for c in timeband_order if c in tb_pivot.columns])

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(tb_pivot, annot=True, fmt='d', cmap='Blues', ax=ax, linewidths=0.5)
ax.set_title('시간대별 주요 OD 쌍 통행량', fontsize=14)
ax.set_xlabel('시간대')
ax.set_ylabel('OD 쌍')
plt.tight_layout()
plt.show()

## 4. 요일별 주요 OD 변화

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_kr = {'Monday': '월', 'Tuesday': '화', 'Wednesday': '수', 'Thursday': '목',
          'Friday': '금', 'Saturday': '토', 'Sunday': '일'}

dow_od = df_top_od.groupby(['day_name', 'OD']).size().reset_index(name='count')
dow_pivot = dow_od.pivot_table(index='OD', columns='day_name', values='count', fill_value=0).astype(int)
dow_pivot = dow_pivot.reindex(columns=[d for d in day_order if d in dow_pivot.columns])
dow_pivot.columns = [day_kr.get(c, c) for c in dow_pivot.columns]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(dow_pivot, annot=True, fmt='d', cmap='Greens', ax=ax, linewidths=0.5)
ax.set_title('요일별 주요 OD 쌍 통행량', fontsize=14)
ax.set_xlabel('요일')
ax.set_ylabel('OD 쌍')
plt.tight_layout()
plt.show()

## 5. 유입/유출 불균형 분석

In [ ]:
# 승차(유출) 건수
outflow = df.groupby('RIDE_A_CD').size().rename('outflow')
# 하차(유입) 건수
inflow = df.groupby('ALIGHT_A_CD').size().rename('inflow')

balance = pd.DataFrame({'outflow': outflow, 'inflow': inflow}).fillna(0).astype(int)
balance['net'] = balance['inflow'] - balance['outflow']  # 양수 = 유입 우세
balance = balance.sort_values('net')

# 상위/하위 각 10개
top_inflow = balance.nlargest(10, 'net')
top_outflow = balance.nsmallest(10, 'net')
balance_plot = pd.concat([top_outflow, top_inflow])

fig, ax = plt.subplots(figsize=(12, 7))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in balance_plot['net']]
ax.barh(range(len(balance_plot)), balance_plot['net'].values, color=colors, edgecolor='black', linewidth=0.5)
ax.set_yticks(range(len(balance_plot)))
ax.set_yticklabels(balance_plot.index, fontsize=9)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('순 유입량 (유입 - 유출)')
ax.set_title('행정동별 유입/유출 불균형 (상위/하위 각 10개)', fontsize=14)
plt.tight_layout()
plt.show()

print('\n--- 유입 우세 행정동 Top 10 ---')
print(top_inflow[['inflow', 'outflow', 'net']].to_string())
print('\n--- 유출 우세 행정동 Top 10 ---')
print(top_outflow[['inflow', 'outflow', 'net']].to_string())

## 6. OD 거리 분포 (평균 이동거리 by 행정동쌍)

In [ ]:
# OD별 평균 운행거리
od_dist = df.groupby('OD').agg(
    mean_dist=('RIDE_DIST', 'mean'),
    trip_count=('RIDE_DIST', 'count')
).reset_index()

# 통행 건수 일정 이상인 OD만 (안정적 평균)
min_trips = max(5, od_dist['trip_count'].quantile(0.5))
od_dist_valid = od_dist[od_dist['trip_count'] >= min_trips].copy()
od_dist_valid = od_dist_valid.sort_values('mean_dist', ascending=False)

# 전체 거리 분포 히스토그램
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) 전체 운행거리 분포
axes[0].hist(df['RIDE_DIST'].dropna(), bins=50, color='steelblue', edgecolor='black', linewidth=0.5)
axes[0].set_title('전체 운행거리 분포')
axes[0].set_xlabel('운행거리 (m)')
axes[0].set_ylabel('건수')

# (b) OD쌍별 평균 거리 분포
axes[1].hist(od_dist_valid['mean_dist'], bins=30, color='coral', edgecolor='black', linewidth=0.5)
axes[1].set_title(f'OD쌍별 평균 운행거리 분포 (N>={int(min_trips)})')
axes[1].set_xlabel('평균 운행거리 (m)')
axes[1].set_ylabel('OD 쌍 수')

plt.tight_layout()
plt.show()

# 장거리/단거리 OD 쌍
print('--- 장거리 OD Top 10 ---')
print(od_dist_valid.head(10)[['OD', 'mean_dist', 'trip_count']].to_string(index=False))
print('\n--- 단거리 OD Top 10 ---')
print(od_dist_valid.tail(10)[['OD', 'mean_dist', 'trip_count']].to_string(index=False))

## 7. 요약 테이블

In [ ]:
summary = {
    '전체 통행 건수': f"{len(df):,}",
    '고유 승차 행정동 수': f"{df['RIDE_A_CD'].nunique():,}",
    '고유 하차 행정동 수': f"{df['ALIGHT_A_CD'].nunique():,}",
    '고유 OD 쌍 수': f"{od_matrix['RIDE_A_CD'].astype(str).str.cat(od_matrix['ALIGHT_A_CD'].astype(str)).nunique():,}",
    '최다 OD 쌍': od_top20.iloc[0]['OD'],
    '최다 OD 통행 건수': f"{od_top20.iloc[0]['trip_count']:,}",
    '평균 운행거리 (m)': f"{df['RIDE_DIST'].mean():,.0f}",
    '유입 최다 행정동': top_inflow.index[0],
    '유출 최다 행정동': top_outflow.index[0],
}

summary_df = pd.DataFrame(list(summary.items()), columns=['항목', '값'])
print('=== OD 매트릭스 분석 요약 ===')
summary_df